##### Copyright 2026 Google LLC.

In [2]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<a href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/examples/Governed_Function_Calling_With_PII_And_Cost_Guardrails.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Governed function calling with PII and cost guardrails

This example shows you how to add deterministic governance guardrails around Gemini's function calling capabilities using [TealTiger](https://github.com/agentguard-ai/tealtiger) (Apache 2.0).

When Gemini agents make autonomous tool calls, you often need to:
- **Block PII** from leaking into tool arguments (SSNs, credit cards, emails)
- **Restrict which tools** the agent can call (allowlist)
- **Enforce cost budgets** per session
- **Produce audit trails** for compliance (SOC2, HIPAA, EU AI Act)

All governance runs deterministically (regex + policy rules) — no additional LLM call, under 2ms overhead per decision.

In [ ]:
%pip install -U -q "google-genai>=2.9.0" tealtiger

In [3]:
from google.colab import userdata
from google import genai

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

C:\Users\satis\AppData\Roaming\Python\Python314\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [4]:
MODEL_ID = "gemini-3.8-flash"  # @param ["gemini-3.5-flash-lite", "gemini-3.5-flash", "gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash", "gemini-2.5-pro", "gemini-3.1-pro-preview"] {"allow-input": true, "isTemplate": true}

## Define tools for the agent

You'll create a simple agent with two tools: `search_database` and `send_email`. In production, you want governance to ensure:
- No PII (like SSNs) gets passed to search queries
- Only authorized tools are callable
- Cost doesn't exceed budget

In [5]:
# Define the tools the agent can call
def search_database(query: str) -> str:
    """Search the internal database for information."""
    return f"Results for '{query}': [Employee records found]"


def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to}: {subject}"


def delete_records(table: str, condition: str) -> str:
    """Delete records from a database table. DANGEROUS."""
    return f"Deleted from {table} where {condition}"


# Map tool names to callables so governed execution can dispatch by name.
tool_functions = {
    "search_database": search_database,
    "send_email": send_email,
    "delete_records": delete_records,
}

# The Interactions API takes tool declarations as dicts (not Python callables).
tool_declarations = [
    {
        "type": "function",
        "name": "search_database",
        "description": "Search the internal database for information.",
        "parameters": {
            "type": "object",
            "properties": {"query": {"type": "string", "description": "The search query"}},
            "required": ["query"],
        },
    },
    {
        "type": "function",
        "name": "send_email",
        "description": "Send an email to a recipient.",
        "parameters": {
            "type": "object",
            "properties": {
                "to": {"type": "string", "description": "Recipient email address"},
                "subject": {"type": "string", "description": "Email subject"},
                "body": {"type": "string", "description": "Email body"},
            },
            "required": ["to", "subject", "body"],
        },
    },
    {
        "type": "function",
        "name": "delete_records",
        "description": "Delete records from a database table. DANGEROUS.",
        "parameters": {
            "type": "object",
            "properties": {
                "table": {"type": "string", "description": "Table name"},
                "condition": {"type": "string", "description": "Deletion condition"},
            },
            "required": ["table", "condition"],
        },
    },
]


## Set up governance policies

TealTiger evaluates governance deterministically — no LLM in the governance path. You define policies declaratively:

In [6]:
import re
import time
from dataclasses import dataclass, field

from tealtiger import PIIDetectionGuardrail, PolicyMode

# The governance building blocks, straight from the published tealtiger package:
#   - PIIDetectionGuardrail: deterministic PII scan over text (async .evaluate()).
#   - PolicyMode: ENFORCE (block), MONITOR (record only), REPORT_ONLY.
# Tool allowlisting, secret detection, and the session budget are small
# deterministic checks you compose around them.

pii_guardrail = PIIDetectionGuardrail()
pii_guardrail.configure({"action": "block"})

# Only these tools may run; delete_records is intentionally left out.
ALLOWED_TOOLS = {"search_database", "send_email"}

# A couple of high-signal secret patterns to catch credentials in tool arguments.
SECRET_PATTERNS = [
    re.compile(r"\bsk-[A-Za-z0-9]{20,}\b"),        # OpenAI-style keys
    re.compile(r"\bAKIA[0-9A-Z]{16}\b"),           # AWS access key IDs
    re.compile(r"\bghp_[A-Za-z0-9]{36}\b"),        # GitHub tokens
]

# Cap cumulative session cost; each governed call is estimated at a flat rate.
COST_PER_CALL = 0.03
SESSION_BUDGET = 0.10

# Governance mode: ENFORCE blocks denied calls; MONITOR records but allows.
MODE = PolicyMode.ENFORCE


@dataclass
class Decision:
    """A single governance decision, collected into the audit trail."""

    action: str  # "ALLOW" or "DENY"
    tool_name: str
    reason_codes: list = field(default_factory=list)
    risk_score: int = 0
    evaluation_time_ms: float = 0.0
    cumulative_cost: float = 0.0


# The audit trail: every evaluation, allowed or denied, is recorded here.
decisions: list[Decision] = []
session_cost = 0.0

print(f"Governance mode: {MODE.value}")
print(f"Allowed tools: {sorted(ALLOWED_TOOLS)}")
print(f"Session budget: ${SESSION_BUDGET:.2f}")


Governance mode: ENFORCE
Allowed tools: ['search_database', 'send_email']
Session budget: $0.10


## Governed function calling

You wrap Gemini's function calling with governance. Before each tool call executes, TealTiger evaluates all policies against the tool name and arguments.

In [7]:
import json


async def governed_tool_call(tool_name: str, tool_args: dict) -> str:
    """Evaluate a tool call against governance, then execute it if allowed.

    Runs four deterministic checks before the tool ever executes:
    tool allowlist, PII scan (TealTiger), secret scan, and a session budget.
    Every decision is recorded in the audit trail regardless of outcome.
    """
    global session_cost

    start = time.perf_counter()
    args_text = json.dumps(tool_args)
    reason_codes: list[str] = []
    risk_score = 0

    # 1. Tool allowlist
    if tool_name not in ALLOWED_TOOLS:
        reason_codes.append("TOOL_NOT_ALLOWED")
        risk_score = max(risk_score, 80)

    # 2. PII in arguments (TealTiger's deterministic guardrail)
    pii_result = await pii_guardrail.evaluate(args_text)
    if not pii_result.passed:
        types = ", ".join(d["type"] for d in pii_result.metadata.get("detections", []))
        reason_codes.append(f"PII_DETECTED:{types}")
        risk_score = max(risk_score, pii_result.risk_score)

    # 3. Secrets in arguments
    if any(pattern.search(args_text) for pattern in SECRET_PATTERNS):
        reason_codes.append("SECRET_DETECTED")
        risk_score = max(risk_score, 95)

    # 4. Session budget
    if session_cost + COST_PER_CALL > SESSION_BUDGET:
        reason_codes.append("BUDGET_EXCEEDED")
        risk_score = max(risk_score, 70)

    action = "DENY" if reason_codes else "ALLOW"
    elapsed_ms = (time.perf_counter() - start) * 1000

    # In MONITOR mode, record the denial but let the call through.
    enforced = action == "DENY" and MODE == PolicyMode.ENFORCE

    if action == "ALLOW" or not enforced:
        session_cost += COST_PER_CALL

    decisions.append(
        Decision(
            action=action,
            tool_name=tool_name,
            reason_codes=reason_codes or ["POLICY_ALLOW"],
            risk_score=risk_score,
            evaluation_time_ms=elapsed_ms,
            cumulative_cost=session_cost,
        )
    )

    print(
        f"  Governance: [{action}] tool={tool_name} "
        f"reason={reason_codes or ['POLICY_ALLOW']} "
        f"risk={risk_score} latency={elapsed_ms:.2f}ms"
    )

    if enforced:
        return f"[BLOCKED] Tool '{tool_name}' denied: {', '.join(reason_codes)}"

    result = tool_functions[tool_name](**tool_args)
    return result


## Example 1: Clean tool call (allowed)

A normal search query with no PII — passes all governance checks.

In [8]:
print("--- Clean query (should ALLOW) ---")
result = await governed_tool_call("search_database", {"query": "quarterly revenue 2025"})
print(f"  Result: {result}\n")

--- Clean query (should ALLOW) ---
  Governance: [ALLOW] tool=search_database reason=['POLICY_ALLOW'] risk=0 latency=0.12ms
  Result: Results for 'quarterly revenue 2025': [Employee records found]



## Example 2: PII in arguments (blocked)

A search query containing an SSN — governance blocks it before execution.

In [9]:
print("--- Query with SSN (should DENY) ---")
result = await governed_tool_call("search_database", {"query": "lookup SSN 123-45-6789"})
print(f"  Result: {result}\n")

--- Query with SSN (should DENY) ---
  Governance: [DENY] tool=search_database reason=['PII_DETECTED:ssn'] risk=90 latency=0.16ms
  Result: [BLOCKED] Tool 'search_database' denied: PII_DETECTED:ssn



## Example 3: Unauthorized tool (blocked)

`delete_records` is not in the allowlist — denied regardless of arguments.

In [10]:
print("--- Unauthorized tool (should DENY) ---")
result = await governed_tool_call("delete_records", {"table": "users", "condition": "active=false"})
print(f"  Result: {result}\n")

--- Unauthorized tool (should DENY) ---
  Governance: [DENY] tool=delete_records reason=['TOOL_NOT_ALLOWED'] risk=80 latency=0.09ms
  Result: [BLOCKED] Tool 'delete_records' denied: TOOL_NOT_ALLOWED



## Example 4: Secret in arguments (blocked)

An API key accidentally passed as a tool argument — blocked before it leaks.

In [11]:
print("--- Secret in args (should DENY) ---")
result = await governed_tool_call("send_email", {
    "to": "ops@company.com",
    "subject": "Deploy config",
    "body": "Use this key: sk-abcdefghij1234567890abcdef"
})
print(f"  Result: {result}\n")

--- Secret in args (should DENY) ---
  Governance: [DENY] tool=send_email reason=['PII_DETECTED:email', 'SECRET_DETECTED'] risk=95 latency=0.11ms
  Result: [BLOCKED] Tool 'send_email' denied: PII_DETECTED:email, SECRET_DETECTED



## Full agent loop with Gemini

Now you wire governance into a real Gemini agent conversation with function calling. After a tool call passes governance and runs, you send its result back to Gemini with `previous_interaction_id` so the model can complete the turn and produce a final answer.

In [ ]:
# Call Gemini through the Interactions API, offering the tool declarations.
# With interactions.create you handle function calls manually, which is exactly
# where governance belongs: evaluate each requested call before executing it.
interaction = client.interactions.create(
    model=MODEL_ID,
    input="Search our database for Q3 2025 revenue numbers",
    tools=tool_declarations,
)

# The model returns function_call steps you handle yourself. Route each one
# through governance before running the underlying tool, then collect the
# results so you can hand them back to the model. Each function_call step
# exposes `id`, `name`, and `arguments`.
called_a_tool = False
function_results = []
for step in interaction.steps:
    if step.type == "function_call":
        called_a_tool = True
        print(f"\nGemini wants to call: {step.name}({dict(step.arguments)})")
        result = await governed_tool_call(step.name, dict(step.arguments))
        print(f"Result: {result}")
        function_results.append(
            {
                "type": "function_result",
                "id": step.id,
                "name": step.name,
                "result": result,
            }
        )

# Complete the multi-turn loop: send the tool result(s) back to Gemini using
# previous_interaction_id so the model can produce its final answer. The
# governance decision (allow or block) is already baked into `result`.
if called_a_tool:
    follow_up = client.interactions.create(
        model=MODEL_ID,
        previous_interaction_id=interaction.id,
        input=function_results,
        tools=tool_declarations,
    )
    if follow_up.output_text:
        print(f"\nGemini's final answer: {follow_up.output_text}")
elif interaction.output_text:
    print(f"Gemini response: {interaction.output_text}")


## Inspect the audit trail

Every governance decision is recorded. This structured evidence is what compliance teams need for SOC2/HIPAA audits.

In [13]:
print("=== Governance Audit Trail ===")
print(f"Total decisions: {len(decisions)}")
print(f"Denials: {sum(1 for d in decisions if d.action == 'DENY')}")
print(f"Session cost: ${session_cost:.4f}")
print()
for i, decision in enumerate(decisions):
    print(
        f"  [{i + 1}] {decision.action} | tool={decision.tool_name} | "
        f"reason={decision.reason_codes} | "
        f"risk={decision.risk_score} | "
        f"latency={decision.evaluation_time_ms:.2f}ms"
    )


=== Governance Audit Trail ===
Total decisions: 5
Denials: 3
Session cost: $0.0600

  [1] ALLOW | tool=search_database | reason=['POLICY_ALLOW'] | risk=0 | latency=0.12ms
  [2] DENY | tool=search_database | reason=['PII_DETECTED:ssn'] | risk=90 | latency=0.16ms
  [3] DENY | tool=delete_records | reason=['TOOL_NOT_ALLOWED'] | risk=80 | latency=0.09ms
  [4] DENY | tool=send_email | reason=['PII_DETECTED:email', 'SECRET_DETECTED'] | risk=95 | latency=0.11ms
  [5] ALLOW | tool=search_database | reason=['POLICY_ALLOW'] | risk=0 | latency=0.08ms


## Governance modes

TealTiger supports three modes for safe rollout:

| Mode | Behavior |
|------|----------|
| **ENFORCE** | Evaluates policies, blocks violations |
| **MONITOR** | Evaluates policies, records decisions, allows all through (dry run) |
| **OBSERVE** | Skips evaluation, passes through with minimal audit |

Start with MONITOR in staging, then switch to ENFORCE in production.

In [14]:
# Switch to MONITOR mode: evaluate and record, but do not block.
MODE = PolicyMode.MONITOR

print("--- MONITOR mode: PII query evaluated but allowed ---")
result = await governed_tool_call("search_database", {"query": "lookup SSN 987-65-4321"})
print(f"  Result: {result}")
print(f"  (Recorded decision: {decisions[-1].action} - would block in ENFORCE)")

# Restore ENFORCE for any later cells.
MODE = PolicyMode.ENFORCE


--- MONITOR mode: PII query evaluated but allowed ---
  Governance: [DENY] tool=search_database reason=['PII_DETECTED:ssn'] risk=90 latency=0.09ms
  Result: Results for 'lookup SSN 987-65-4321': [Employee records found]
  (Recorded decision: DENY - would block in ENFORCE)


## Summary

In this notebook, you learned how to:
1. Add PII detection to Gemini function call arguments
2. Restrict which tools the agent can call via allowlists
3. Enforce per-session cost budgets
4. Detect leaked secrets before they reach external systems
5. Produce structured audit trails for compliance

All governance is deterministic — no additional LLM call, under 2ms per decision.

## What's next

- Learn more about [function calling in Gemini](https://ai.google.dev/gemini-api/docs/function-calling)
- Explore [Gemini agent capabilities](https://ai.google.dev/gemini-api/docs/agents)
- See [TealTiger policy templates](https://docs.tealtiger.ai/policy-templates) for SOC2, HIPAA, GDPR presets
- Try [governance modes](https://docs.tealtiger.ai) — start with MONITOR, promote to ENFORCE
- Read more about [Gemini API pricing](https://ai.google.dev/pricing) to set appropriate budgets